In [ ]:
from flask import Flask, request, render_template, redirect, send_from_directory
import torch
import os
import cv2
import numpy as np
from ultralytics import YOLO
from concurrent.futures import ThreadPoolExecutor, as_completed
from fuzzywuzzy import fuzz
from transformers import VisionEncoderDecoderModel, ViTImageProcessor, TrOCRProcessor, ViTFeatureExtractor, NllbTokenizer
import asyncio
from multiprocessing import cpu_count
import unicodedata
import time
import math
import matplotlib.pyplot as plt

In [39]:


# Check if GPU is available and set device accordingly
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load YOLOv8 model (Load once when the server starts) and move to GPU/CPU
detection_model = YOLO('train18/weights/best.pt').to(device)

# # Load Processor (Load once when the server starts)
# tokenizer = NllbTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
# feature_extractor = ViTFeatureExtractor.from_pretrained('microsoft/trocr-small-printed')
# processor = TrOCRProcessor(feature_extractor=feature_extractor, tokenizer=tokenizer)
# Load TRoCR model (Load once when the server starts) and move to GPU/CPU
# recognition_model = VisionEncoderDecoderModel.from_pretrained('small-printed-v2/checkpoint-37640').to(device)
recognition_model = VisionEncoderDecoderModel.from_pretrained('checkpoint-119112').to(device)
recognition_model.eval()

processor = TrOCRProcessor.from_pretrained('checkpoint-119112')


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [37]:
import cv2
import os
import asyncio
import torch
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor, as_completed
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

def preprocess_image(image_path):
    image = cv2.imread(image_path)
    if image is None:
        raise ValueError(f"Image not found at path: {image_path}")
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Convert to RGB
    return image

def detect_text(image, model):
    results = model(image)
    boxes = [box for box, conf in zip(results[0].boxes.xyxy.cpu().numpy(), results[0].boxes.conf.cpu().numpy()) if conf >= 0.3]
    return boxes

def group_boxes_by_lines(boxes, iou_threshold=0.5):
    """Group bounding boxes by lines based on their IoU."""
    lines = []
    boxes = [list(box) for box in boxes]  # Ensure boxes are in list format

    while boxes:
        current_box = boxes.pop(0)
        line_group = [current_box]

        for other_box in boxes[:]:
            if calculate_iou(current_box, other_box) > iou_threshold:
                line_group.append(other_box)
                boxes.remove(other_box)

        lines.append(line_group)

    return lines

def concatenate_boxes(line_group):
    """Concatenate bounding boxes into a single box covering the entire line."""
    x_min = min([box[0] for box in line_group])
    y_min = min([box[1] for box in line_group])
    x_max = max([box[2] for box in line_group])
    y_max = max([box[3] for box in line_group])

    return [x_min, y_min, x_max, y_max]

async def recognize_text_by_lines(image, line_groups, model, processor):
    recognized_texts = []

    for line_group in line_groups:
        # Concatenate all bounding boxes in the current line group
        line_box = concatenate_boxes(line_group)
        x1, y1, x2, y2 = map(int, line_box)
        roi = image[y1:y2, x1:x2]
        roi = cv2.resize(roi, (400, 400))
        pixel_values = processor(images=roi, return_tensors="pt").pixel_values.to(device)
        generated_ids = model.generate(pixel_values, max_new_tokens=50)
        recognized_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        recognized_texts.append(recognized_text)
    
    return recognized_texts

def detect_and_recognize(image, detection_model, recognition_model, processor):
    detections = detect_text(image, detection_model)
    detections.sort(key=lambda box: (box[1], box[0]))  # Sort by y and x to ensure top-to-bottom, left-to-right order
    line_groups = group_boxes_by_lines(detections)
    
    # Parallelize recognition over line groups
    futures = []
    with ThreadPoolExecutor(max_workers=cpu_count()) as executor:
        for line_group in line_groups:
            futures.append(executor.submit(asyncio.run, recognize_text_by_lines(image, [line_group], recognition_model, processor)))
    
    recognized_texts = []
    for future in as_completed(futures):
        recognized_texts.extend(future.result())
    
    return recognized_texts

def process_image(image_path, detection_model, recognition_model, processor):
    image = preprocess_image(image_path)
    recognized_texts = detect_and_recognize(image, detection_model, recognition_model, processor)
    
    # Print the recognized texts in the desired format
    print("['" + "', '".join(recognized_texts) + "']")
    
    # Plot the image using matplotlib
    plt.imshow(image)
    plt.axis('off')  # Turn off axis labels
    plt.show()

def normalize_text(text):
    # Implement any text normalization you need
    return text


In [40]:


def preprocess_image(image_path):
    image = cv2.imread(image_path)
    if image is None:
        raise ValueError(f"Image not found at path: {image_path}")
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Convert to RGB
    
    # Apply preprocessing: binarization and noise reduction
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    _, binary_image = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    binary_image = cv2.fastNlMeansDenoising(binary_image, None, 30, 7, 21)
    
    return binary_image

def detect_text(image, model):
    results = model(image)
    boxes = [box for box, conf in zip(results[0].boxes.xyxy.cpu().numpy(), results[0].boxes.conf.cpu().numpy()) if conf >= 0.3]
    return boxes

def group_boxes_by_lines(boxes, iou_threshold=0.5):
    """Group bounding boxes by lines based on their IoU."""
    lines = []
    boxes = [list(box) for box in boxes]  # Ensure boxes are in list format

    while boxes:
        current_box = boxes.pop(0)
        line_group = [current_box]

        for other_box in boxes[:]:
            if calculate_iou(current_box, other_box) > iou_threshold:
                line_group.append(other_box)
                boxes.remove(other_box)

        lines.append(line_group)

    return lines

def concatenate_boxes(line_group):
    """Concatenate bounding boxes into a single box covering the entire line."""
    x_min = min([box[0] for box in line_group])
    y_min = min([box[1] for box in line_group])
    x_max = max([box[2] for box in line_group])
    y_max = max([box[3] for box in line_group])

    return [x_min, y_min, x_max, y_max]

async def recognize_text_by_lines(image, line_groups, model, processor):
    recognized_texts = []

    for line_group in line_groups:
        # Concatenate all bounding boxes in the current line group
        line_box = concatenate_boxes(line_group)
        x1, y1, x2, y2 = map(int, line_box)
        roi = image[y1:y2, x1:x2]
        roi = cv2.resize(roi, (384, 384))
        pixel_values = processor(images=roi, return_tensors="pt").pixel_values.to(device)
        generated_ids = model.generate(pixel_values, max_new_tokens=50)
        recognized_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        recognized_text = normalize_text_khmer(recognized_text)  # Normalize Khmer text
        recognized_texts.append(recognized_text)
    
    return recognized_texts

def detect_and_recognize(image, detection_model, recognition_model, processor):
    detections = detect_text(image, detection_model)
    detections.sort(key=lambda box: (box[1], box[0]))  # Sort by y and x to ensure top-to-bottom, left-to-right order
    line_groups = group_boxes_by_lines(detections)
    
    # Parallelize recognition over line groups
    futures = []
    with ThreadPoolExecutor(max_workers=cpu_count()) as executor:
        for line_group in line_groups:
            futures.append(executor.submit(asyncio.run, recognize_text_by_lines(image, [line_group], recognition_model, processor)))
    
    recognized_texts = []
    for future in as_completed(futures):
        recognized_texts.extend(future.result())
    
    return recognized_texts

def process_image(image_path, detection_model, recognition_model, processor):
    image = preprocess_image(image_path)
    recognized_texts = detect_and_recognize(image, detection_model, recognition_model, processor)
    
    # Print the recognized texts in the desired format
    print("['" + "', '".join(recognized_texts) + "']")
    
    # Plot the image using matplotlib
    plt.imshow(image, cmap='gray')  # Show the binary image
    plt.axis('off')  # Turn off axis labels
    plt.show()

def normalize_text_khmer(text):
    # Implement Khmer-specific text normalization here
    # For example, remove unnecessary spaces, correct common OCR errors, etc.
    return text




In [41]:
# Process an image
image_path = 'images/kh_data_8794.png'
recognized_texts = process_image(image_path, detection_model, recognition_model, processor)

ValueError: axes don't match array

In [ ]:
# # Process an image
# image_path = 'images/kh_data_10025.png'
# recognized_texts = process_image(image_path, detection_model, recognition_model, processor)
# print("All recognized texts:")
# for _, text in recognized_texts:
#     print(f"- {text}")